In [3]:
# ============================================================
# CREDIT RISK PREDICTION
# File: preprocessing.py
#
# Purpose:
# - Load data
# - Feature engineering
# - Prepare target variable
# - Train/test split
# - Create preprocessing pipeline
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)


# ============================================================
# 2. FEATURE ENGINEERING
# ============================================================

def create_features(df):

    df = df.copy()

    # Credit amount relative to loan duration
    df["Credit_Amount_per_Duration"] = (
        df["Credit_Amount"] / df["Duration"]
    )

    # Credit amount relative to applicant age
    df["Credit_Amount_per_Age"] = (
        df["Credit_Amount"] / df["Age"]
    )

    # Loan duration relative to applicant age
    df["Duration_Age_Ratio"] = (
        df["Duration"] / df["Age"]
    )

    # Credit amount relative to existing credits
    df["Credit_Amount_per_Existing_Credit"] = (
        df["Credit_Amount"] /
        (df["Existing_Credits"] + 1)
    )

    return df


# ============================================================
# 3. PREPARE DATA
# ============================================================

def prepare_data(df):

    # Apply feature engineering
    df = create_features(df)

    # Convert target:
    # 1 = Good Risk → 0
    # 2 = Bad Risk  → 1
    df["Credit_Risk"] = df["Credit_Risk"].map({
        1: 0,
        2: 1
    })

    # Separate features and target
    X = df.drop(
        "Credit_Risk",
        axis=1
    )

    y = df["Credit_Risk"]

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    return (
        X_train,
        X_test,
        y_train,
        y_test
    )


# ============================================================
# 4. NUMERICAL FEATURES
# ============================================================

NUMERICAL_FEATURES = [

    "Duration",
    "Credit_Amount",
    "Installment_Rate",
    "Residence_Since",
    "Age",
    "Existing_Credits",
    "Liable_People",

    # Engineered features
    "Credit_Amount_per_Duration",
    "Credit_Amount_per_Age",
    "Duration_Age_Ratio",
    "Credit_Amount_per_Existing_Credit"
]


# ============================================================
# 5. CATEGORICAL FEATURES
# ============================================================

CATEGORICAL_FEATURES = [

    "Status_Checking",
    "Credit_History",
    "Purpose",
    "Savings",
    "Employment",
    "Personal_Status_Sex",
    "Other_Debtors",
    "Property",
    "Other_Installment_Plans",
    "Housing",
    "Job",
    "Telephone",
    "Foreign_Worker"
]


# ============================================================
# 6. NUMERICAL PREPROCESSING
# ============================================================

numerical_pipeline = Pipeline(
    steps=[

        # Handle missing numerical values
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        # Standardise numerical features
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ============================================================
# 7. CATEGORICAL PREPROCESSING
# ============================================================

categorical_pipeline = Pipeline(
    steps=[

        # Handle missing categorical values
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        # Convert categories into numerical features
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


# ============================================================
# 8. COMPLETE PREPROCESSING PIPELINE
# ============================================================

preprocessor = ColumnTransformer(

    transformers=[

        (
            "numerical",
            numerical_pipeline,
            NUMERICAL_FEATURES
        ),

        (
            "categorical",
            categorical_pipeline,
            CATEGORICAL_FEATURES
        )
    ]
)


# ============================================================
# 9. TEST THE PREPROCESSING FILE
# ============================================================

if __name__ == "__main__":

    # Import only the data-loading function
    from data_eda import load_data

    # Load dataset
    df = load_data()

    # Prepare data
    (
        X_train,
        X_test,
        y_train,
        y_test
    ) = prepare_data(df)

    print("=" * 60)
    print("PREPROCESSING CHECK")
    print("=" * 60)

    print("\nOriginal dataset shape:")
    print(df.shape)

    print("\nTraining data shape:")
    print(X_train.shape)

    print("\nTesting data shape:")
    print(X_test.shape)

    print("\nTraining target distribution:")
    print(y_train.value_counts())

    print("\nTesting target distribution:")
    print(y_test.value_counts())

    print("\nPreprocessor created successfully!")

    print("\nPreprocessing pipeline:")
    print(preprocessor)

ModuleNotFoundError: No module named 'data_eda'